> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTA-js2I/lm1uXy0dKQBcrruDsbJt5w/view?utm_content=DAGzTA-js2I&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=hf88292377f)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. 模型调用


## 2.1 AI Studio 模型调用

由于 AI Studio 并未区分 LangChain 的社区版本和官方专门版本，但考虑到绝大多数模型均兼容 OpenAI 的调用格式，因此后续我们将统一使用 **langchain-openai** 库进行模型调用，以保证调用方式的简洁性、兼容性和维护的便利性。

### 2.1.1 直接调用

可通过 .invoke 来传入对话信息：

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
  model="ernie-3.5-8k",
  openai_api_key=os.environ.get("OPENAI_API_KEY"),
  base_url="https://aistudio.baidu.com/llm/lmapi/v3" 
)

response = llm.invoke("你好，请介绍一下你自己")

print(response.content)

### 2.1.2 传入 OpenAI 格式内容
也可以直接传入上下文内容进行调用，比如这里是 OpenAI 格式的内容：

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="ernie-3.5-8k",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3")

conversation = [{"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}]

response = llm.invoke(conversation)
print(response.content)

### 2.1.3 传入 LangChain 格式内容
另一种是传入 langchain 官方的 AIMessage、HumanMessage 以及 SystemMessage 等内容：

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os

llm = ChatOpenAI(model="ernie-3.5-8k",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3")

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")]

response = llm.invoke(conversation)
print(response.content)

### 2.1.4 同时调用多个提问
假如一次性传入多个提问可使用 .batch 实现（仅支持 openai 这种专用库实现）：

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="ernie-3.5-8k",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3")

question = [ "Why do parrots have colorful feathers?", "How do airplanes fly?", "What is quantum computing?"]

responses = llm.batch(question)
for response in responses: 
    print(response)

默认情况下只会返回最终的结果，但假如希望获取完整的调用信息，可以使用 .batch_as_completed 的方法实现： 

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="ernie-3.5-8k",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3")

question = [ "Why do parrots have colorful feathers?", "How do airplanes fly?", "What is quantum computing?"]

responses = llm.batch_as_completed(question)
for response in responses:
    print(response)

### 2.1.5 Content Blocks

目前这个功能仅支持 langchain-openai 或 langchain-anthropic 等专用包。

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
 model="ernie-5.0-thinking-preview",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3" 
)

response = llm.invoke("你好，请介绍一下你自己")
print(response.content_blocks)

此外，在最新版的 LangChain 中还推出了 init_chat_model 来统一大量模型的入口，不过当前该功能也仅支持专用包的模型调用。

In [ ]:
from langchain.chat_models import init_chat_model

# o3_mini = init_chat_model("openai:o3-mini", temperature=0)
# claude_sonnet = init_chat_model("anthropic:claude-sonnet-4-5-20250929", temperature=0)
# gemini_2_5_flash = init_chat_model("google_vertexai:gemini-2.5-flash", temperature=0)

# o3_mini.invoke("what's your name")
# claude_sonnet.invoke("what's your name")
# gemini_2_5_flash.invoke("what's your name")

## 3.3 LangChain-Community 模型调用

并不是每个模型都有自己单独的包，大部分都是国外的大厂（比如 OpenAI、Google、Anthropic 等），而国内只有 Deepseek 有，并且支持力度也相对较低。

而更多其他的模型是通过 langchain-community 来实现支持，比如前面提到的通义千问，这也是国内厂商里与 LangChain 适配程度比较高的模型（前提是根据第一章的指引获取到了 DASHSCOPE_API_KEY ：

In [ ]:
from langchain_community.chat_models import ChatTongyi
import os

# 不需要写入 url，已在内部默认写入
llm = ChatTongyi(model="qwen-max",api_key=os.environ.get("DASHSCOPE_API_KEY"))  # 替换为你的 DashScope API Key

response = llm.invoke("你好，请介绍一下你自己")
print(response.content)